In [115]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# SUBJECT DATA

In [116]:
subject_data = pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_1y_progress_DATA/Data_Subject_1Year_12SEP2025.csv')
subject_data.drop(columns=['Unnamed: 0','DATE_VISIT'], inplace=True) # Date_visit no nos da informacion relevante es mas informacion estructural para el dataset
print(subject_data.shape)
subject_data.head()

(911, 34)


,PATNO,Visit ID,Height (cm),Weight (kg),Temperature (Celsius),Supine BP - systolic (mmHg),Supine BP - diastolic (mmHg),Supine heart rate (bpm),Standing BP - systolic (mmHg),Standing BP - diastolic (mmHg),...,Identify self as White,Number of years of education reported by the participant,AGE_AT_VISIT,MDS-UPDRS Total Score 1YearProg,MDS-UPDRS Total Score,Δ TOTAL_UPDRS_1Y,Δ UPDRS_I_1Y,Δ UPDRS_II_1Y,Δ UPDRS_III_1Y,Δ UPDRS_IV_1Y
0,100006,V04_V06,180.0,115.0,36.6,131.0,84.0,66.0,123.0,84.0,...,1,15.0,56.7,34.0,36.0,-2.0,4,-4.0,-8,6.0
1,100007,V04_V06,172.0,68.0,36.3,128.0,64.0,65.0,120.0,82.0,...,1,19.0,68.4,19.0,32.0,-13.0,0,-7.0,-7,1.0
2,100012,V04_V06,159.0,78.7,36.4,132.0,84.0,59.0,131.0,75.0,...,1,18.0,67.0,33.0,25.0,8.0,1,4.0,3,0.0
3,100018,V04_V06,167.0,59.2,36.5,108.0,68.0,67.0,112.0,77.0,...,1,18.0,71.0,59.0,44.0,15.0,0,1.0,14,0.0
4,101018,V04_V06,175.0,81.1,36.7,116.0,76.0,66.0,105.0,73.0,...,1,16.0,63.2,23.0,27.0,-4.0,2,-1.0,-3,-2.0


## Col correction
- Birth date 
- años de estudio
- progressiontype

In [117]:
# 1. Change of format of Birth Date
subject_data['Birth Date'] = pd.to_datetime(subject_data['Birth Date'], errors='coerce')
subject_data['Birth Date'] = subject_data['Birth Date'].dt.year
subject_data['Birth Date'] = subject_data['Birth Date'].astype('Int64')

# Create Birth Cohort variable
subject_data["Birth Cohort"] = pd.cut(
    subject_data["Birth Date"],
    bins=4,               # 4 cortes con igual rango
    labels=[1, 2, 3, 4],  # etiquetas ordinales
    include_lowest=True
) #IntervalIndex([(1932.946, 1943.8],   (1943.8, 1954.6],   (1954.6, 1965.4],(1965.4, 1976.2],   (1976.2, 1987.0]],dtype='interval[float64, right]') 

subject_data.drop(columns=['Birth Date'], inplace=True)

# 2. Change column name to 'Education_Years'
old_col = [col for col in subject_data.columns if 'years' in col.lower()][0]

subject_data = subject_data.rename(columns={
    old_col: 'Education_Years'
})

# 3. change delta porgression columns
def categorize_progression(delta):
    if delta <= -5:
        return 2 #'Substantial improv.'
    elif -5 < delta <= 0:
        return 1 #'Slight improv.'
    elif 0 < delta <= 9:
        return -1 #'Mild progress.'
    else:
        return -2 #'Substantial progress.'


subject_data['ProgressionType'] = subject_data['Δ TOTAL_UPDRS_1Y'].apply(categorize_progression)
subject_data.drop(columns=['Δ TOTAL_UPDRS_1Y'], inplace=True)



## New cols
- BMI
- Ortostatic_Hypotension

In [118]:
# 1. corrección de alturas erróneas y cálculo de BMI
subject_data.loc[subject_data['PATNO'] == 73935, 'Height (cm)'] = 177
subject_data.loc[subject_data['PATNO'] == 127741, 'Height (cm)'] = 186
subject_data["BMI"] = subject_data["Weight (kg)"] / ( (subject_data["Height (cm)"] / 100) ** 2 )

# 2. Deltas posturales
subject_data["Delta_SBP"] = subject_data["Standing BP - systolic (mmHg)"] - subject_data["Supine BP - systolic (mmHg)"]
subject_data["Delta_DBP"] = subject_data["Standing BP - diastolic (mmHg)"] - subject_data["Supine BP - diastolic (mmHg)"]
subject_data["Delta_HR"]  = subject_data["Standing heart rate (bpm)"] - subject_data["Supine heart rate (bpm)"]




## Drop non relevant cols

In [119]:
# 1. cols that has an unique value

list_drop_columns = []
for col in subject_data.columns:
    if subject_data[col].nunique()== 1:
        list_drop_columns.append(col)#RBD at Enrollment/Pink1 Mutation at Enrollmen/Identify self as Hawaiian/Other Pacific Islander

subject_data.drop(columns=list_drop_columns, inplace=True)

# 2. eliminamos columna altura y peso debido a que ya tenemos el BMI

subject_data.drop(columns=['Height (cm)','Weight (kg)'], inplace=True)

# 3. eliminamos columnas relacionadas con Delta_SBP, Delta_DBP y Delta_HR

subject_data.drop(columns=[
    'Standing BP - systolic (mmHg)','Supine BP - systolic (mmHg)',
    'Standing BP - diastolic (mmHg)','Supine BP - diastolic (mmHg)',
    'Standing heart rate (bpm)','Supine heart rate (bpm)'
], inplace=True)

# progresion futura valor de MDS-UPDRS total

subject_data.drop(columns=['MDS-UPDRS Total Score 1YearProg'],inplace=True)


In [120]:
subject_data.shape

(911, 26)

In [121]:
subject_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/MODELS DEVELOPMENT/FINAL_DATA/Subject_1Year_Data.csv')

# MOTOR DATA

In [122]:
motor_data = pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_1y_progress_DATA/Data_Motor_1Year_12SEP2025.csv')
motor_data.drop(columns=['Unnamed: 0','Is the participant on medication or receiving deep brain stimulation for treating the symptoms of Parkinsons disease?',
                            'Is subject on medication for PD'],inplace=True)
LEDD_DATA=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_1y_progress_DATA/Data_Medication_1Year_12SEP2025.csv')
LEDD_DATA=LEDD_DATA[['PATNO','Visit ID','Average LEDD Dose']]
# Merge datasets
motor_data = motor_data.merge(LEDD_DATA, on=['PATNO','Visit ID'], how='inner')
motor_data



,PATNO,Visit ID,SCHWAB & ENGLAND ADL,MDS-UPDRS Part I (Patient Questionnaire) Total Score,MDS-UPDRS Part II Total Score,Does participant have DBS,MDS-UPDRS Part III Total Score,3.21 HOEHN AND YAHR STAGE,MDS-UPDRS Part IV Total Score,DBS_Transition_Visit,DBS_Post_Transition,MDS-UPDRS Total Score 1YearProg,MDS-UPDRS Total Score,Δ TOTAL_UPDRS_1Y,Δ UPDRS_I_1Y,Δ UPDRS_II_1Y,Δ UPDRS_III_1Y,Δ UPDRS_IV_1Y,Average LEDD Dose
0,100006,V04_V06,95,8,9.0,0,17,2,2.0,False,0.0,34.0,36.0,-2.0,4,-4.0,-8,6.0,0.00
1,100007,V04_V06,95,2,7.0,0,23,2,0.0,False,0.0,19.0,32.0,-13.0,0,-7.0,-7,1.0,337.50
2,100012,V04_V06,95,6,5.0,0,14,1,0.0,False,0.0,33.0,25.0,8.0,1,4.0,3,0.0,300.00
3,100018,V04_V06,90,7,17.0,0,17,2,3.0,False,0.0,59.0,44.0,15.0,0,1.0,14,0.0,450.00
4,101018,V04_V06,95,6,5.0,0,14,2,2.0,False,0.0,23.0,27.0,-4.0,2,-1.0,-3,-2.0,600.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
906,75480,V10_V12,90,14,12.0,0,20,2,3.0,False,0.0,32.0,49.0,-17.0,-6,-5.0,-3,-3.0,1017.50
907,75484,V10_V12,95,5,8.0,0,16,2,5.0,False,0.0,34.0,34.0,0.0,0,-5.0,3,2.0,466.25
908,75505,V10_V12,90,16,12.0,1,32,2,3.0,True,0.0,66.0,63.0,3.0,-8,8.0,-8,11.0,515.00
909,75524,V10_V12,100,11,7.0,0,17,2,5.0,False,0.0,44.0,40.0,4.0,-1,0.0,10,-5.0,875.00


## Col correction
-  DBS_Transition_Visit

In [123]:
motor_data['DBS_Transition_Visit'] = motor_data['DBS_Transition_Visit'].map({True: 1, False: 0})
motor_data['ProgressionType'] = motor_data['Δ TOTAL_UPDRS_1Y'].apply(categorize_progression)
motor_data.drop(columns=['Δ TOTAL_UPDRS_1Y'], inplace=True)
motor_data

,PATNO,Visit ID,SCHWAB & ENGLAND ADL,MDS-UPDRS Part I (Patient Questionnaire) Total Score,MDS-UPDRS Part II Total Score,Does participant have DBS,MDS-UPDRS Part III Total Score,3.21 HOEHN AND YAHR STAGE,MDS-UPDRS Part IV Total Score,DBS_Transition_Visit,DBS_Post_Transition,MDS-UPDRS Total Score 1YearProg,MDS-UPDRS Total Score,Δ UPDRS_I_1Y,Δ UPDRS_II_1Y,Δ UPDRS_III_1Y,Δ UPDRS_IV_1Y,Average LEDD Dose,ProgressionType
0,100006,V04_V06,95,8,9.0,0,17,2,2.0,0,0.0,34.0,36.0,4,-4.0,-8,6.0,0.00,1
1,100007,V04_V06,95,2,7.0,0,23,2,0.0,0,0.0,19.0,32.0,0,-7.0,-7,1.0,337.50,2
2,100012,V04_V06,95,6,5.0,0,14,1,0.0,0,0.0,33.0,25.0,1,4.0,3,0.0,300.00,-1
3,100018,V04_V06,90,7,17.0,0,17,2,3.0,0,0.0,59.0,44.0,0,1.0,14,0.0,450.00,-2
4,101018,V04_V06,95,6,5.0,0,14,2,2.0,0,0.0,23.0,27.0,2,-1.0,-3,-2.0,600.00,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
906,75480,V10_V12,90,14,12.0,0,20,2,3.0,0,0.0,32.0,49.0,-6,-5.0,-3,-3.0,1017.50,2
907,75484,V10_V12,95,5,8.0,0,16,2,5.0,0,0.0,34.0,34.0,0,-5.0,3,2.0,466.25,1
908,75505,V10_V12,90,16,12.0,1,32,2,3.0,1,0.0,66.0,63.0,-8,8.0,-8,11.0,515.00,-1
909,75524,V10_V12,100,11,7.0,0,17,2,5.0,0,0.0,44.0,40.0,-1,0.0,10,-5.0,875.00,-1


## Drop non relevant cols

In [124]:
motor_data.drop(columns=['MDS-UPDRS Total Score 1YearProg',], inplace=True)
motor_data

,PATNO,Visit ID,SCHWAB & ENGLAND ADL,MDS-UPDRS Part I (Patient Questionnaire) Total Score,MDS-UPDRS Part II Total Score,Does participant have DBS,MDS-UPDRS Part III Total Score,3.21 HOEHN AND YAHR STAGE,MDS-UPDRS Part IV Total Score,DBS_Transition_Visit,DBS_Post_Transition,MDS-UPDRS Total Score,Δ UPDRS_I_1Y,Δ UPDRS_II_1Y,Δ UPDRS_III_1Y,Δ UPDRS_IV_1Y,Average LEDD Dose,ProgressionType
0,100006,V04_V06,95,8,9.0,0,17,2,2.0,0,0.0,36.0,4,-4.0,-8,6.0,0.00,1
1,100007,V04_V06,95,2,7.0,0,23,2,0.0,0,0.0,32.0,0,-7.0,-7,1.0,337.50,2
2,100012,V04_V06,95,6,5.0,0,14,1,0.0,0,0.0,25.0,1,4.0,3,0.0,300.00,-1
3,100018,V04_V06,90,7,17.0,0,17,2,3.0,0,0.0,44.0,0,1.0,14,0.0,450.00,-2
4,101018,V04_V06,95,6,5.0,0,14,2,2.0,0,0.0,27.0,2,-1.0,-3,-2.0,600.00,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
906,75480,V10_V12,90,14,12.0,0,20,2,3.0,0,0.0,49.0,-6,-5.0,-3,-3.0,1017.50,2
907,75484,V10_V12,95,5,8.0,0,16,2,5.0,0,0.0,34.0,0,-5.0,3,2.0,466.25,1
908,75505,V10_V12,90,16,12.0,1,32,2,3.0,1,0.0,63.0,-8,8.0,-8,11.0,515.00,-1
909,75524,V10_V12,100,11,7.0,0,17,2,5.0,0,0.0,40.0,-1,0.0,10,-5.0,875.00,-1


In [125]:
motor_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/MODELS DEVELOPMENT/FINAL_DATA/Motor_1Year_Data.csv')

# COGNITIVE DATA

In [126]:
cognitive_data = pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_1y_progress_DATA/Data_Cognitive_1Year_12SEP2025.csv')
cognitive_data.drop(columns=['Unnamed: 0','MDS-UPDRS Part I (Patient Questionnaire) Total Score','MDS-UPDRS Part II Total Score','MDS-UPDRS Part III Total Score'], inplace=True)
print(cognitive_data.shape)


(911, 35)


## Col correction
-  Motor Exam assessment
- Coordination assessment
- Sensory Exam assessment
- Reflexes assessment
- CN II-XII assessment
- Δ TOTAL_UPDRS_1Y

In [127]:
cog_obj_list=[]
for col in cognitive_data.columns[3:]:
    if cognitive_data[col].dtype == 'object':
        cog_obj_list.append(col)

print(cog_obj_list)

for col in cog_obj_list:
    cognitive_data[col]=cognitive_data[col].map({'Normal': 1, 'Abnormal': 0,'Unable to test': 0,'Not tested':0})


cognitive_data['Has Depression']=cognitive_data['Has Depression'].map({True: 1, False: 0})

cognitive_data['ProgressionType'] = cognitive_data['Δ TOTAL_UPDRS_1Y'].apply(categorize_progression)
cognitive_data.drop(columns=['Δ TOTAL_UPDRS_1Y'], inplace=True)
cognitive_data

['Motor Exam assessment', 'Coordination assessment', 'Sensory Exam assessment', 'Reflexes assessment', 'CN II-XII assessment']


,PATNO,Visit ID,STAI PART I Score,STAI PART II Score,STAI Total Score,Semantic Fluency Scaled Score,HVLT Total Recall Score,HVLT Delayed Recall Score,HVLT Retention Score,HVLT Recognition Discrimination Index Score,...,Reflexes assessment,CN II-XII assessment,Benton Judgement of Line Orientation Scaled Score,MDS-UPDRS Total Score 1YearProg,MDS-UPDRS Total Score,Δ UPDRS_I_1Y,Δ UPDRS_II_1Y,Δ UPDRS_III_1Y,Δ UPDRS_IV_1Y,ProgressionType
0,100006,V04_V06,48.0,38.0,86.0,11,56.0,55.0,55.0,44.0,...,1,1,13.49,34.0,36.0,4,-4.0,-8,6.0,1
1,100007,V04_V06,49.0,47.0,96.0,18,42.0,40.0,36.0,25.0,...,1,1,11.47,19.0,32.0,0,-7.0,-7,1.0,2
2,100012,V04_V06,49.0,49.0,98.0,8,40.0,53.0,50.0,52.0,...,1,1,12.80,33.0,25.0,1,4.0,3,0.0,-1
3,100018,V04_V06,44.0,41.0,85.0,11,48.0,63.0,66.0,59.0,...,1,1,15.00,59.0,44.0,0,1.0,14,0.0,-2
4,101018,V04_V06,48.0,45.0,93.0,13,22.0,27.0,36.0,52.0,...,1,1,6.66,23.0,27.0,2,-1.0,-3,-2.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
906,75480,V10_V12,47.0,44.0,91.0,8,39.0,40.0,41.0,41.0,...,1,1,10.83,32.0,49.0,-6,-5.0,-3,-3.0,2
907,75484,V10_V12,46.0,41.0,87.0,17,55.0,45.0,38.0,56.0,...,1,1,10.14,34.0,34.0,0,-5.0,3,2.0,1
908,75505,V10_V12,57.0,47.0,104.0,9,46.0,53.0,50.0,56.0,...,1,1,12.34,66.0,63.0,-8,8.0,-8,11.0,-1
909,75524,V10_V12,53.0,43.0,96.0,12,27.0,20.0,20.0,33.0,...,1,1,11.06,44.0,40.0,-1,0.0,10,-5.0,-1


## Drop non relevant cols

In [128]:
cognitive_data.drop(columns=['MDS-UPDRS Total Score 1YearProg'], inplace=True)
cognitive_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 911 entries, 0 to 910
Data columns (total 34 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   PATNO                                              911 non-null    int64  
 1   Visit ID                                           911 non-null    object 
 2   STAI PART I Score                                  911 non-null    float64
 3   STAI PART II Score                                 911 non-null    float64
 4   STAI Total Score                                   911 non-null    float64
 5   Semantic Fluency Scaled Score                      911 non-null    int64  
 6   HVLT Total Recall Score                            911 non-null    float64
 7   HVLT Delayed Recall Score                          911 non-null    float64
 8   HVLT Retention Score                               911 non-null    float64
 9   HVLT Recog

In [129]:
cognitive_data.isna().sum()

PATNO                                                0
Visit ID                                             0
STAI PART I Score                                    0
STAI PART II Score                                   0
STAI Total Score                                     0
Semantic Fluency Scaled Score                        0
HVLT Total Recall Score                              0
HVLT Delayed Recall Score                            0
HVLT Retention Score                                 0
HVLT Recognition Discrimination Index Score          0
GDS Short Score                                      0
Has Depression                                       0
MoCA Total Score                                     0
VISUOSPATIAL_EXECUTIVE                               0
NAMING                                               0
ATTENTION                                            0
LANGUAGE                                             0
ABSTRACTION                                          0
DELAYED_RE

In [130]:
cognitive_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/MODELS DEVELOPMENT/FINAL_DATA/Cognitive_1Year_Data.csv')

# SLEEP DATA

In [131]:
sleep_data = pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_1y_progress_DATA/Data_Sleep_1Year_12SEP2025.csv')
sleep_data.drop(columns=['Unnamed: 0','MDS-UPDRS Part I (Patient Questionnaire) Total Score','MDS-UPDRS Part II Total Score','MDS-UPDRS Part III Total Score'], inplace=True)
print(sleep_data.shape)


(911, 38)


## Col correction and Drop non relevant cols
- Δ TOTAL_UPDRS_1Y

In [132]:
sleep_data['ProgressionType'] = sleep_data['Δ TOTAL_UPDRS_1Y'].apply(categorize_progression)
sleep_data.drop(columns=['Δ TOTAL_UPDRS_1Y','MDS-UPDRS Total Score 1YearProg'], inplace=True)
sleep_data.head()

,PATNO,Visit ID,Sitting and reading,Watching TV,"Sitting, inactive in a public place",As a passenger in a car for an hour,Lying down to rest in the afternoon,Sitting and talking to someone,Sitting quietly after lunch,"In a car, while stopped in traffic",...,Narcolepsy,Depression,Epilepsy,Inflammatory disease of the brain,MDS-UPDRS Total Score,Δ UPDRS_I_1Y,Δ UPDRS_II_1Y,Δ UPDRS_III_1Y,Δ UPDRS_IV_1Y,ProgressionType
0,100006,V04_V06,0,1,0.0,1,2,0.0,0,0,...,0,1,0,0,36.0,4,-4.0,-8,6.0,1
1,100007,V04_V06,1,0,0.0,0,1,0.0,0,0,...,0,0,0,0,32.0,0,-7.0,-7,1.0,2
2,100012,V04_V06,1,1,0.0,0,1,0.0,1,0,...,0,0,0,0,25.0,1,4.0,3,0.0,-1
3,100018,V04_V06,0,0,0.0,1,1,0.0,0,0,...,0,0,0,0,44.0,0,1.0,14,0.0,-2
4,101018,V04_V06,1,1,0.0,1,1,0.0,0,0,...,0,0,0,0,27.0,2,-1.0,-3,-2.0,1


In [133]:
sleep_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/MODELS DEVELOPMENT/FINAL_DATA/Sleep_1Year_Data.csv')

# MEDICATION DATA

In [134]:
med_data = pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_1y_progress_DATA/Data_Medication_1Year_12SEP2025.csv')
med_data.drop(columns=['Unnamed: 0','MDS-UPDRS Part I (Patient Questionnaire) Total Score','MDS-UPDRS Part II Total Score','MDS-UPDRS Part III Total Score'], inplace=True)
print(med_data.shape)
med_data.head()

(911, 22)


,PATNO,Visit ID,Cardiovascular,Digestiva,Preventiva / Suplementos,Neurológica / Psiquiátrica,Endocrina / Metabólica,Urinaria / Genitourinaria,Musculoesquelética / Dolor,Dermatológica,...,NO PD Medication,Active LEDD Medication at visit,Average LEDD Dose,MDS-UPDRS Total Score 1YearProg,MDS-UPDRS Total Score,Δ TOTAL_UPDRS_1Y,Δ UPDRS_I_1Y,Δ UPDRS_II_1Y,Δ UPDRS_III_1Y,Δ UPDRS_IV_1Y
0,100006,V04_V06,1,0,2,2,1,0,2,0,...,1,Ninguna,0.0,34.0,36.0,-2.0,4,-4.0,-8,6.0
1,100007,V04_V06,1,0,0,0,0,0,0,0,...,1,Carbidopa/Levodopa CR,337.5,19.0,32.0,-13.0,0,-7.0,-7,1.0
2,100012,V04_V06,2,1,1,0,1,0,0,0,...,1,Sinemet (Carbidopa/Levodopa IR),300.0,33.0,25.0,8.0,1,4.0,3,0.0
3,100018,V04_V06,0,0,0,0,0,0,1,0,...,1,Carbidopa/Levodopa ER + Carbidopa/Levodopa IR,450.0,59.0,44.0,15.0,0,1.0,14,0.0
4,101018,V04_V06,0,0,0,0,1,1,0,0,...,1,Carbidopa/Levodopa IR,600.0,23.0,27.0,-4.0,2,-1.0,-3,-2.0


## Col correction and Drop non relevant cols
- Δ TOTAL_UPDRS_1Y

In [135]:
med_data['ProgressionType'] = med_data['Δ TOTAL_UPDRS_1Y'].apply(categorize_progression)
med_data.drop(columns=['Δ TOTAL_UPDRS_1Y','MDS-UPDRS Total Score 1YearProg'], inplace=True)
med_data

,PATNO,Visit ID,Cardiovascular,Digestiva,Preventiva / Suplementos,Neurológica / Psiquiátrica,Endocrina / Metabólica,Urinaria / Genitourinaria,Musculoesquelética / Dolor,Dermatológica,...,Ginecológica / Hormonales,NO PD Medication,Active LEDD Medication at visit,Average LEDD Dose,MDS-UPDRS Total Score,Δ UPDRS_I_1Y,Δ UPDRS_II_1Y,Δ UPDRS_III_1Y,Δ UPDRS_IV_1Y,ProgressionType
0,100006,V04_V06,1,0,2,2,1,0,2,0,...,0,1,Ninguna,0.00,36.0,4,-4.0,-8,6.0,1
1,100007,V04_V06,1,0,0,0,0,0,0,0,...,0,1,Carbidopa/Levodopa CR,337.50,32.0,0,-7.0,-7,1.0,2
2,100012,V04_V06,2,1,1,0,1,0,0,0,...,0,1,Sinemet (Carbidopa/Levodopa IR),300.00,25.0,1,4.0,3,0.0,-1
3,100018,V04_V06,0,0,0,0,0,0,1,0,...,0,1,Carbidopa/Levodopa ER + Carbidopa/Levodopa IR,450.00,44.0,0,1.0,14,0.0,-2
4,101018,V04_V06,0,0,0,0,1,1,0,0,...,0,1,Carbidopa/Levodopa IR,600.00,27.0,2,-1.0,-3,-2.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
906,75480,V10_V12,2,2,0,0,8,2,2,0,...,0,1,Amantadin (Amantadine) + Carbidopa/Levodopa ER...,1017.50,49.0,-6,-5.0,-3,-3.0,2
907,75484,V10_V12,0,0,3,0,0,0,2,0,...,0,1,Azilect (Rasagiline) + Rytary (Carbidopa/Levod...,466.25,34.0,0,-5.0,3,2.0,1
908,75505,V10_V12,1,2,1,0,1,0,0,0,...,0,1,Amantadin (Amantadine) + Carbidopa/Levodopa IR...,515.00,63.0,-8,8.0,-8,11.0,-1
909,75524,V10_V12,1,0,0,0,1,1,0,0,...,0,1,Amantadine + Carbidopa/Levodopa IR + Pramipexo...,875.00,40.0,-1,0.0,10,-5.0,-1


- Active LEDD Medication at visit

In [136]:
import re
from collections import Counter

frases = med_data['Active LEDD Medication at visit'].to_list()

# Unir todas las frases en un solo texto
texto = " ".join(frases)

# Extraer solo palabras ASCII (a-z)
palabras = re.findall(r"\b[a-zA-Z]+\b", texto.lower())

# Filtro: palabras con más de 3 letras y sin números (ya garantizado por el regex)
palabras_filtradas = [p for p in palabras if len(p) == 2]

# Contar frecuencia
contador = Counter(palabras_filtradas)

# Mostrar todas las palabras ordenadas por frecuencia
mas_comunes = contador.most_common()
print(mas_comunes)

[('ir', 296), ('er', 140), ('cr', 66), ('mg', 23), ('xl', 21), ('pk', 16), ('co', 11), ('lt', 7), ('sr', 1)]


In [137]:
from sklearn.preprocessing import MultiLabelBinarizer
# ================================================================
# 2) DICCIONARIO DE NORMALIZACIÓN (TODO EN LISTAS)
# ================================================================
normalizacion = {

    # LEVODOPA
    'levodopa': ['levodopa'],
    'levadopa': ['levodopa'],
    'levidopa': ['levodopa'],
    'levodpoav': ['levodopa'],
    'levodpa': ['levodopa'],
    'levodop': ['levodopa'],
    'levopdopa': ['levodopa'],
    'loevadopa': ['levodopa'],
    'levodapa': ['levodopa'],
    'ldopa': ['levodopa'],

    # CARBIDOPA
    'carbidopa': ['carbidopa'],
    'cabidopa': ['carbidopa'],
    'carbdiopa': ['carbidopa'],
    'carb': ['carbidopa'],

    # BENSERAZIDE
    'benserazide': ['benserazide'],
    'benserazid': ['benserazide'],
    'benzerazide': ['benserazide'],
    'benseracide': ['benserazide'],

    # LEVODOPA + CARBIDOPA (MARCAS)
    'sinemet': ['levodopa', 'carbidopa'],
    'sinement': ['levodopa', 'carbidopa'],
    'rytary': ['levodopa', 'carbidopa'],
    'dopicar': ['levodopa', 'carbidopa'],
    'levocomp': ['levodopa', 'carbidopa'],
    'careldopa': ['levodopa', 'carbidopa'],

    # LEVODOPA + BENSERAZIDE
    'madopar': ['levodopa', 'benserazide'],
    'modutab': ['levodopa', 'benserazide'],
    'prolopa': ['levodopa', 'benserazide'],
    'isicom': ['levodopa', 'benserazide'],
    'levopar': ['levodopa', 'benserazide'],

    # PRAMIPEXOL
    'pramipexole': ['pramipexole'],
    'pramipexol': ['pramipexole'],
    'pramiprexole': ['pramipexole'],
    'mirapex': ['pramipexole'],
    'mirapexin': ['pramipexole'],
    'sifrol': ['pramipexole'],
    'oprymea': ['pramipexole'],

    # ROPINIROLE
    'ropinirole': ['ropinirole'],
    'ropinerole': ['ropinirole'],
    'ropinirol': ['ropinirole'],
    'ropinrole': ['ropinirole'],
    'requip': ['ropinirole'],

    # ROTIGOTINA
    'rotigotine': ['rotigotine'],
    'rotigotin': ['rotigotine'],
    'neupro': ['rotigotine'],

    # RASAGILINA
    'rasagiline': ['rasagiline'],
    'rasagilina': ['rasagiline'],
    'rasagaline': ['rasagiline'],
    'rasageline': ['rasagiline'],
    'rasagilene': ['rasagiline'],
    'rasagilin': ['rasagiline'],
    'azilect': ['rasagiline'],

    # SAFINAMIDA
    'safinamide': ['safinamide'],
    'safinamida': ['safinamide'],
    'xadago': ['safinamide'],

    # AMANTADINA
    'amantadine': ['amantadine'],
    'amantadin': ['amantadine'],
    'symmetrel': ['amantadine'],
    'gocovri': ['amantadine'],

    # APOMORFINA
    'apomorphine': ['apomorphine'],
    'apomorfin': ['apomorphine'],

    # OTROS
    'selegiline': ['selegiline'],
    'selegilene': ['selegiline'],
    'piribedil': ['piribedil'],
    'trihexyphenidyl': ['trihexyphenidyl'],
    'artane': ['trihexyphenidyl'],

    # LEVODOPA INHALADA
    'inbrija': ['levodopa'],

    # formas de liberacion
    "ir": ["Immediate Release"],
    "cr": ["Controlled Release"],
    "er": ["Extended Release"]
}


# ================================================================
# 3) FUNCIÓN DE LIMPIEZA Y NORMALIZACIÓN
# ================================================================
def limpiar_y_normalizar(texto):
    if pd.isna(texto):
        return []

    t = texto.lower()
    t = re.sub(r"[+/()\-]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()

    tokens = t.split()
    activos = []

    for tok in tokens:
        if tok in normalizacion:
            activos.extend(normalizacion[tok])

    return sorted(set(activos))  # sin duplicados


# ================================================================
# 4) APLICAR LIMPIEZA A TODA LA COLUMNA
# ================================================================
med_data['lista_meds'] = med_data['Active LEDD Medication at visit'].apply(limpiar_y_normalizar)


# ================================================================
# 5) CREAR DUMMIES
# ================================================================
mlb = MultiLabelBinarizer()
dummies = pd.DataFrame(
    mlb.fit_transform(med_data['lista_meds']),
    columns=mlb.classes_,
    index=med_data.index
)

med_data = pd.concat([med_data, dummies], axis=1)
med_data.head()




,PATNO,Visit ID,Cardiovascular,Digestiva,Preventiva / Suplementos,Neurológica / Psiquiátrica,Endocrina / Metabólica,Urinaria / Genitourinaria,Musculoesquelética / Dolor,Dermatológica,...,carbidopa,levodopa,piribedil,pramipexole,rasagiline,ropinirole,rotigotine,safinamide,selegiline,trihexyphenidyl
0,100006,V04_V06,1,0,2,2,1,0,2,0,...,0,0,0,0,0,0,0,0,0,0
1,100007,V04_V06,1,0,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
2,100012,V04_V06,2,1,1,0,1,0,0,0,...,1,1,0,0,0,0,0,0,0,0
3,100018,V04_V06,0,0,0,0,0,0,1,0,...,1,1,0,0,0,0,0,0,0,0
4,101018,V04_V06,0,0,0,0,1,1,0,0,...,1,1,0,0,0,0,0,0,0,0


- generic medication

In [138]:
def classification(x):
    return 0 if x == 0 else 1

col_not_PD_med=['Cardiovascular', 'Digestiva',
              'Preventiva / Suplementos', 'Neurológica / Psiquiátrica',
              'Endocrina / Metabólica', 'Urinaria / Genitourinaria',
              'Musculoesquelética / Dolor', 'Dermatológica', 'Infecciosa',
              'Ginecológica / Hormonales']

for col in col_not_PD_med:
    med_data[col]=med_data[col].astype(int).apply(classification)

med_data.drop(columns=['Active LEDD Medication at visit','lista_meds', ], inplace=True)
med_data

,PATNO,Visit ID,Cardiovascular,Digestiva,Preventiva / Suplementos,Neurológica / Psiquiátrica,Endocrina / Metabólica,Urinaria / Genitourinaria,Musculoesquelética / Dolor,Dermatológica,...,carbidopa,levodopa,piribedil,pramipexole,rasagiline,ropinirole,rotigotine,safinamide,selegiline,trihexyphenidyl
0,100006,V04_V06,1,0,1,1,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,100007,V04_V06,1,0,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
2,100012,V04_V06,1,1,1,0,1,0,0,0,...,1,1,0,0,0,0,0,0,0,0
3,100018,V04_V06,0,0,0,0,0,0,1,0,...,1,1,0,0,0,0,0,0,0,0
4,101018,V04_V06,0,0,0,0,1,1,0,0,...,1,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
906,75480,V10_V12,1,1,0,0,1,1,1,0,...,1,1,0,0,0,0,0,0,0,0
907,75484,V10_V12,0,0,1,0,0,0,1,0,...,1,1,0,0,1,0,0,0,0,0
908,75505,V10_V12,1,1,1,0,1,0,0,0,...,1,1,0,0,0,1,0,0,0,0
909,75524,V10_V12,1,0,0,0,1,1,0,0,...,1,1,0,1,0,0,0,0,0,0


In [139]:
med_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/MODELS DEVELOPMENT/FINAL_DATA/Medication_1Year_Data.csv')

# AE DATA


In [140]:
ae_data=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_1y_progress_DATA/Data_Adverse_Events_1Year_12SEP2025.csv')
ae_data.drop(columns=['Unnamed: 0','MDS-UPDRS Part I (Patient Questionnaire) Total Score','MDS-UPDRS Part II Total Score','MDS-UPDRS Part III Total Score'], inplace=True)
ae_data

,PATNO,Visit ID,Any adverse events observed?,LP performed on assessment date,Skin Biopsy performed on assessment date,Dopamine Imaging performed on assessment date,Total_AE,Mild_AE,Moderate_AE,Severe_AE,...,R_Possible,R_LP,R_BiopsySkin,MDS-UPDRS Total Score 1YearProg,MDS-UPDRS Total Score,Δ TOTAL_UPDRS_1Y,Δ UPDRS_I_1Y,Δ UPDRS_II_1Y,Δ UPDRS_III_1Y,Δ UPDRS_IV_1Y
0,100006,V04_V06,No,Unchecked,Unchecked,Unchecked,0,0,0,0,...,0,0,0,34.0,36.0,-2.0,4,-4.0,-8,6.0
1,100007,V04_V06,No,Unchecked,Unchecked,Unchecked,0,0,0,0,...,0,0,0,19.0,32.0,-13.0,0,-7.0,-7,1.0
2,100012,V04_V06,No,Unchecked,Unchecked,Unchecked,0,0,0,0,...,0,0,0,33.0,25.0,8.0,1,4.0,3,0.0
3,100018,V04_V06,No,Unchecked,Unchecked,Unchecked,0,0,0,0,...,0,0,0,59.0,44.0,15.0,0,1.0,14,0.0
4,101018,V04_V06,No,Unchecked,Unchecked,Unchecked,0,0,0,0,...,0,0,0,23.0,27.0,-4.0,2,-1.0,-3,-2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
906,75480,V10_V12,No,Unchecked,Unchecked,Unchecked,1,0,1,0,...,0,1,0,32.0,49.0,-17.0,-6,-5.0,-3,-3.0
907,75484,V10_V12,No,Unchecked,Unchecked,Unchecked,0,0,0,0,...,0,0,0,34.0,34.0,0.0,0,-5.0,3,2.0
908,75505,V10_V12,No,Unchecked,Unchecked,Unchecked,0,0,0,0,...,0,0,0,66.0,63.0,3.0,-8,8.0,-8,11.0
909,75524,V10_V12,No,Unchecked,Unchecked,Unchecked,0,0,0,0,...,0,0,0,44.0,40.0,4.0,-1,0.0,10,-5.0


## Col correction and Drop non relevant cols
- Δ TOTAL_UPDRS_1Y

In [141]:
ae_data['ProgressionType'] = ae_data['Δ TOTAL_UPDRS_1Y'].apply(categorize_progression)
ae_data.drop(columns=['Δ TOTAL_UPDRS_1Y','MDS-UPDRS Total Score 1YearProg','R_BiopsySkin','R_LP'], inplace=True)
ae_data

,PATNO,Visit ID,Any adverse events observed?,LP performed on assessment date,Skin Biopsy performed on assessment date,Dopamine Imaging performed on assessment date,Total_AE,Mild_AE,Moderate_AE,Severe_AE,R_Definite_AE,R_Probable_AE,R_Possible,MDS-UPDRS Total Score,Δ UPDRS_I_1Y,Δ UPDRS_II_1Y,Δ UPDRS_III_1Y,Δ UPDRS_IV_1Y,ProgressionType
0,100006,V04_V06,No,Unchecked,Unchecked,Unchecked,0,0,0,0,0,0,0,36.0,4,-4.0,-8,6.0,1
1,100007,V04_V06,No,Unchecked,Unchecked,Unchecked,0,0,0,0,0,0,0,32.0,0,-7.0,-7,1.0,2
2,100012,V04_V06,No,Unchecked,Unchecked,Unchecked,0,0,0,0,0,0,0,25.0,1,4.0,3,0.0,-1
3,100018,V04_V06,No,Unchecked,Unchecked,Unchecked,0,0,0,0,0,0,0,44.0,0,1.0,14,0.0,-2
4,101018,V04_V06,No,Unchecked,Unchecked,Unchecked,0,0,0,0,0,0,0,27.0,2,-1.0,-3,-2.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
906,75480,V10_V12,No,Unchecked,Unchecked,Unchecked,1,0,1,0,1,0,0,49.0,-6,-5.0,-3,-3.0,2
907,75484,V10_V12,No,Unchecked,Unchecked,Unchecked,0,0,0,0,0,0,0,34.0,0,-5.0,3,2.0,1
908,75505,V10_V12,No,Unchecked,Unchecked,Unchecked,0,0,0,0,0,0,0,63.0,-8,8.0,-8,11.0,-1
909,75524,V10_V12,No,Unchecked,Unchecked,Unchecked,0,0,0,0,0,0,0,40.0,-1,0.0,10,-5.0,-1


In [142]:
ae_data.columns
ae_data['Any adverse events observed?']=ae_data['Any adverse events observed?'].replace({'Yes': 1, 'No': 0})

cols = [
    'LP performed on assessment date',
    'Skin Biopsy performed on assessment date',
    'Dopamine Imaging performed on assessment date'
]

ae_data[cols] = ae_data[cols].replace({'Checked': 1, 'Unchecked': 0})
ae_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 911 entries, 0 to 910
Data columns (total 19 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   PATNO                                          911 non-null    int64  
 1   Visit ID                                       911 non-null    object 
 2   Any adverse events observed?                   911 non-null    int64  
 3   LP performed on assessment date                911 non-null    int64  
 4   Skin Biopsy performed on assessment date       911 non-null    int64  
 5   Dopamine Imaging performed on assessment date  911 non-null    int64  
 6   Total_AE                                       911 non-null    int64  
 7   Mild_AE                                        911 non-null    int64  
 8   Moderate_AE                                    911 non-null    int64  
 9   Severe_AE                                      911 non

/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_82146/2771182072.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ae_data['Any adverse events observed?']=ae_data['Any adverse events observed?'].replace({'Yes': 1, 'No': 0})
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_82146/2771182072.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ae_data[cols] = ae_data[cols].replace({'Checked': 1, 'Unchecked': 0})


In [143]:
ae_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/MODELS DEVELOPMENT/FINAL_DATA/Adverse_Events_1Year_Data.csv')